# Tavsiye Sistemleri (Recommendation Systems)

Bu çalışmada, tavsiye sistemleri hakkındaki öğrendiğimiz 2 temel yöntem olan **Content Based** ve **Collaborative Filtering**'in uygulamada nasıl çalıştıklarını göreceğiz.

Bu dersin sonunda şunları yapabiliyor olmayı hedefliyoruz:

1. Bir tavsiye sistemi oluşturmak için **hangi bilgilerin gerekli olduğunu** kavramak
2. **Content Based** & **Collaborative** yaklaşımlarının farklılıklarını anlamak
3. Kullanıcılar veya ürünler arasındaki benzerliklere tespit etmeye çalışırken kullanılan **mesafe metriklerini** anlamak
4. Ürün veya içerik **önerisinde nasıl bulunabileceğimizi** görmek

## 1. Content Based Tavsiye Sistemleri: Biralar

Günlük hayatımızda birçok defa bu tarz durumlarla karşılaşıyoruz. Bir yiyeceği, içeceği veya ürünü beğeniyoruz ve ona **benzer özelliklerde olan diğer ürünleri** araştırmaya başlıyoruz. Sonuç kimi zaman başarılı kimi zaman ise hüsranla sonuçlanabiliyor :(

O zaman hadi bu örneğimizde de, biralar üzerinde yapacağımız incelemelerle **birbirine benzer özelliklerde olan biraları** bulmaya çalışalım.

![Beers](https://253qv1sx4ey389p9wtpp9sj0-wpengine.netdna-ssl.com/wp-content/uploads/2020/09/BEER_Styles_Credit_HERO_FALLBACK_1920x1280.jpg)

Tabiki öncelikle gerekli **kütüphanelerimizi** ve **veri setimizi** yükleyelim.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('/Users/ataozarslan/Downloads/beer_reviews.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1586614 entries, 0 to 1586613
Data columns (total 13 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   brewery_id          1586614 non-null  int64  
 1   brewery_name        1586599 non-null  object 
 2   review_time         1586614 non-null  int64  
 3   review_overall      1586614 non-null  float64
 4   review_aroma        1586614 non-null  float64
 5   review_appearance   1586614 non-null  float64
 6   review_profilename  1586266 non-null  object 
 7   beer_style          1586614 non-null  object 
 8   review_palate       1586614 non-null  float64
 9   review_taste        1586614 non-null  float64
 10  beer_name           1586614 non-null  object 
 11  beer_abv            1518829 non-null  float64
 12  beer_beerid         1586614 non-null  int64  
dtypes: float64(6), int64(3), object(4)
memory usage: 157.4+ MB


Bazı sütunlarımızda **boş veriler** var. Eğer o sütunları kullanmamız gerekirse, gerekli düzenlemeleri yaparız. Şu anlık bu şekilde kalabilir. Biz veri setimizi biraz daha anlamaya çalışalım.

In [3]:
df.head()

,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,beer_style,review_palate,review_taste,beer_name,beer_abv,beer_beerid
0,10325,Vecchio Birraio,1234817823,1.5,2.0,2.5,stcules,Hefeweizen,1.5,1.5,Sausa Weizen,5.0,47986
1,10325,Vecchio Birraio,1235915097,3.0,2.5,3.0,stcules,English Strong Ale,3.0,3.0,Red Moon,6.2,48213
2,10325,Vecchio Birraio,1235916604,3.0,2.5,3.0,stcules,Foreign / Export Stout,3.0,3.0,Black Horse Black Beer,6.5,48215
3,10325,Vecchio Birraio,1234725145,3.0,3.0,3.5,stcules,German Pilsener,2.5,3.0,Sausa Pils,5.0,47969
4,1075,Caldera Brewing Company,1293735206,4.0,4.5,4.0,johnmichaelsen,American Double / Imperial IPA,4.0,4.5,Cauldron DIPA,7.7,64883


İlk aşamada bir **Content Based Filtering** çalışması yapalım. Bu amaç doğrultusunda örneğin, **johnmichaelsen** kullancısının değerlendirmelerini göz önüne alarak ona yeni bira önerilerinde bulunalım :)

In [4]:
df[df.review_profilename.isin(['johnmichaelsen'])]

,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,beer_style,review_palate,review_taste,beer_name,beer_abv,beer_beerid
4,1075,Caldera Brewing Company,1293735206,4.0,4.5,4.0,johnmichaelsen,American Double / Imperial IPA,4.0,4.5,Cauldron DIPA,7.70,64883
2264,21841,Craggie Brewing Company,1275305721,4.0,3.5,4.0,johnmichaelsen,Keller Bier / Zwickel Bier,4.0,3.5,Swannanoa Sunset,4.20,56980
2893,14879,Hoppin' Frog Brewery,1314626098,4.0,4.5,4.0,johnmichaelsen,American Double / Imperial IPA,3.5,4.0,Hop Dam Triple IPA,10.00,56115
3142,14879,Hoppin' Frog Brewery,1229353931,3.5,4.5,4.0,johnmichaelsen,American Double / Imperial IPA,4.0,4.0,Mean Manalishi Double I.P.A.,8.20,37518
3911,14879,Hoppin' Frog Brewery,1197296940,4.0,4.0,4.0,johnmichaelsen,American IPA,4.0,4.5,Hoppin' To Heaven IPA,6.80,33624
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1583321,41,Brouwerij Corsendonk,1197058463,4.0,4.5,4.0,johnmichaelsen,Belgian Strong Dark Ale,4.0,5.0,Corsendonk Christmas Ale,8.50,140
1583794,41,Brouwerij Corsendonk,1189351111,4.0,4.0,4.0,johnmichaelsen,Dubbel,4.0,4.0,Corsendonk Pater / Abbey Brown Ale,7.50,138
1584268,41,Brouwerij Corsendonk,1203370485,4.0,4.0,4.0,johnmichaelsen,Tripel,4.0,4.5,Corsendonk Agnus / Abbey Pale Ale,7.50,139
1585654,3835,Drake's Brewing Co.,1233373447,4.0,4.5,4.0,johnmichaelsen,American Double / Imperial IPA,4.0,4.5,Drake's Denogginizer,9.75,22273


Görünüşe göre **johnmichaelsen** kullanıcısının toplamda **2111 tane** değerlendirmesi var. Bakalım bu kullanıcımız hangi biralar için değerlendirmede bulunmuş? Çünkü tavsiyede bulunurken zaten içtiği bir birayı önermek istemeyiz :)

In [5]:
jm_beer_choices = df[df.review_profilename.isin(['johnmichaelsen'])].beer_style.unique()
len(jm_beer_choices) # 93 farklı bira için değerlendirmede bulunmuş

93

Content Based sistemlerde çalışırken en mantıklı sonuçları elde etmek için, incelediğimiz ürünle ilgili elimizde daha çok bilgi olması gerekir. Ama bakalım bu elimizdeki verilerden ne çıkarabiliyoruz...

>**SORU:** Veri setine baktığınızda benzer biraları bulmak için sizce hangi verilerden yararlanmak en mantıklı seçenek?

Benzer biraları bulurken **bira çeşitlerini** kullanmak oldukça mantıklı bir yaklaşım, ancak şöyle bir sorunumuz var:

- `beer_style` sütundaki bazı satırlarda **birden fazla çeşit** verilmiş, tam doğru sonuçlara ulaşabilmek için bu değerleri biraz düzenlemek gerekecek!

In [6]:
df.beer_style.nunique()

104

In [7]:
# Bir satırda en fazla kaç tane bira çeşidi var?
for sum_style in df.beer_style.str.split('/'):
    if len(sum_style) > 2:
        raise ValueError("2'den fazla bira çeşidi var!") # Kontrol mekanizması :)

Demek ki **maksimum 2 bira çeşidi** verilmiş. Şimdi bu bilgiyi kullanarak düzenlememizi yapalım.

In [8]:
splited_style = df.beer_style.str.split('/', n=1, expand=True)
 
# 1. Bira Türü
df['1st_beer_style']= splited_style[0]
# 2. Bira Türü
df['2nd_beer_style']= splited_style[1]

Artık ihtiyacımız olmadığı için `beer_style` sütununu kaldırabiliriz.

In [9]:
df.drop(columns='beer_style', inplace=True)
df

,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,review_palate,review_taste,beer_name,beer_abv,beer_beerid,1st_beer_style,2nd_beer_style
0,10325,Vecchio Birraio,1234817823,1.5,2.0,2.5,stcules,1.5,1.5,Sausa Weizen,5.0,47986,Hefeweizen,None
1,10325,Vecchio Birraio,1235915097,3.0,2.5,3.0,stcules,3.0,3.0,Red Moon,6.2,48213,English Strong Ale,None
2,10325,Vecchio Birraio,1235916604,3.0,2.5,3.0,stcules,3.0,3.0,Black Horse Black Beer,6.5,48215,Foreign,Export Stout
3,10325,Vecchio Birraio,1234725145,3.0,3.0,3.5,stcules,2.5,3.0,Sausa Pils,5.0,47969,German Pilsener,None
4,1075,Caldera Brewing Company,1293735206,4.0,4.5,4.0,johnmichaelsen,4.0,4.5,Cauldron DIPA,7.7,64883,American Double,Imperial IPA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1586609,14359,The Defiant Brewing Company,1162684892,5.0,4.0,3.5,maddogruss,4.0,4.0,The Horseman's Ale,5.2,33061,Pumpkin Ale,None
1586610,14359,The Defiant Brewing Company,1161048566,4.0,5.0,2.5,yelterdow,2.0,4.0,The Horseman's Ale,5.2,33061,Pumpkin Ale,None
1586611,14359,The Defiant Brewing Company,1160702513,4.5,3.5,3.0,TongoRad,3.5,4.0,The Horseman's Ale,5.2,33061,Pumpkin Ale,None
1586612,14359,The Defiant Brewing Company,1160023044,4.0,4.5,4.5,dherling,4.5,4.5,The Horseman's Ale,5.2,33061,Pumpkin Ale,None


Ek olarak, bu ayrımdan sonra elimizde kaç tane benzersiz bira çeşidi olduğuna bakalım.

In [10]:
print('Unique Beer Style(1st):', df['1st_beer_style'].nunique())
print('Unique Beer Style(2nd):', df['2nd_beer_style'].nunique())

Unique Beer Style(1st): 101
Unique Beer Style(2nd): 19


In [11]:
df

,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,review_palate,review_taste,beer_name,beer_abv,beer_beerid,1st_beer_style,2nd_beer_style
0,10325,Vecchio Birraio,1234817823,1.5,2.0,2.5,stcules,1.5,1.5,Sausa Weizen,5.0,47986,Hefeweizen,None
1,10325,Vecchio Birraio,1235915097,3.0,2.5,3.0,stcules,3.0,3.0,Red Moon,6.2,48213,English Strong Ale,None
2,10325,Vecchio Birraio,1235916604,3.0,2.5,3.0,stcules,3.0,3.0,Black Horse Black Beer,6.5,48215,Foreign,Export Stout
3,10325,Vecchio Birraio,1234725145,3.0,3.0,3.5,stcules,2.5,3.0,Sausa Pils,5.0,47969,German Pilsener,None
4,1075,Caldera Brewing Company,1293735206,4.0,4.5,4.0,johnmichaelsen,4.0,4.5,Cauldron DIPA,7.7,64883,American Double,Imperial IPA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1586609,14359,The Defiant Brewing Company,1162684892,5.0,4.0,3.5,maddogruss,4.0,4.0,The Horseman's Ale,5.2,33061,Pumpkin Ale,None
1586610,14359,The Defiant Brewing Company,1161048566,4.0,5.0,2.5,yelterdow,2.0,4.0,The Horseman's Ale,5.2,33061,Pumpkin Ale,None
1586611,14359,The Defiant Brewing Company,1160702513,4.5,3.5,3.0,TongoRad,3.5,4.0,The Horseman's Ale,5.2,33061,Pumpkin Ale,None
1586612,14359,The Defiant Brewing Company,1160023044,4.0,4.5,4.5,dherling,4.5,4.5,The Horseman's Ale,5.2,33061,Pumpkin Ale,None


In [12]:
df_new = df.copy()
df_new

,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,review_palate,review_taste,beer_name,beer_abv,beer_beerid,1st_beer_style,2nd_beer_style
0,10325,Vecchio Birraio,1234817823,1.5,2.0,2.5,stcules,1.5,1.5,Sausa Weizen,5.0,47986,Hefeweizen,None
1,10325,Vecchio Birraio,1235915097,3.0,2.5,3.0,stcules,3.0,3.0,Red Moon,6.2,48213,English Strong Ale,None
2,10325,Vecchio Birraio,1235916604,3.0,2.5,3.0,stcules,3.0,3.0,Black Horse Black Beer,6.5,48215,Foreign,Export Stout
3,10325,Vecchio Birraio,1234725145,3.0,3.0,3.5,stcules,2.5,3.0,Sausa Pils,5.0,47969,German Pilsener,None
4,1075,Caldera Brewing Company,1293735206,4.0,4.5,4.0,johnmichaelsen,4.0,4.5,Cauldron DIPA,7.7,64883,American Double,Imperial IPA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1586609,14359,The Defiant Brewing Company,1162684892,5.0,4.0,3.5,maddogruss,4.0,4.0,The Horseman's Ale,5.2,33061,Pumpkin Ale,None
1586610,14359,The Defiant Brewing Company,1161048566,4.0,5.0,2.5,yelterdow,2.0,4.0,The Horseman's Ale,5.2,33061,Pumpkin Ale,None
1586611,14359,The Defiant Brewing Company,1160702513,4.5,3.5,3.0,TongoRad,3.5,4.0,The Horseman's Ale,5.2,33061,Pumpkin Ale,None
1586612,14359,The Defiant Brewing Company,1160023044,4.0,4.5,4.5,dherling,4.5,4.5,The Horseman's Ale,5.2,33061,Pumpkin Ale,None


Artık elimizdeki bu bilgileri kullanarak birbirine benzer biraları bulabiliriz :)

In [13]:
df_new['attributes'] = df_new['brewery_name'] + ' ' + df_new['1st_beer_style'] + ' ' + df_new['1st_beer_style']
df_new

,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,review_palate,review_taste,beer_name,beer_abv,beer_beerid,1st_beer_style,2nd_beer_style,attributes
0,10325,Vecchio Birraio,1234817823,1.5,2.0,2.5,stcules,1.5,1.5,Sausa Weizen,5.0,47986,Hefeweizen,None,Vecchio Birraio Hefeweizen Hefeweizen
1,10325,Vecchio Birraio,1235915097,3.0,2.5,3.0,stcules,3.0,3.0,Red Moon,6.2,48213,English Strong Ale,None,Vecchio Birraio English Strong Ale English Str...
2,10325,Vecchio Birraio,1235916604,3.0,2.5,3.0,stcules,3.0,3.0,Black Horse Black Beer,6.5,48215,Foreign,Export Stout,Vecchio Birraio Foreign Foreign
3,10325,Vecchio Birraio,1234725145,3.0,3.0,3.5,stcules,2.5,3.0,Sausa Pils,5.0,47969,German Pilsener,None,Vecchio Birraio German Pilsener German Pilsener
4,1075,Caldera Brewing Company,1293735206,4.0,4.5,4.0,johnmichaelsen,4.0,4.5,Cauldron DIPA,7.7,64883,American Double,Imperial IPA,Caldera Brewing Company American Double Ameri...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1586609,14359,The Defiant Brewing Company,1162684892,5.0,4.0,3.5,maddogruss,4.0,4.0,The Horseman's Ale,5.2,33061,Pumpkin Ale,None,The Defiant Brewing Company Pumpkin Ale Pumpki...
1586610,14359,The Defiant Brewing Company,1161048566,4.0,5.0,2.5,yelterdow,2.0,4.0,The Horseman's Ale,5.2,33061,Pumpkin Ale,None,The Defiant Brewing Company Pumpkin Ale Pumpki...
1586611,14359,The Defiant Brewing Company,1160702513,4.5,3.5,3.0,TongoRad,3.5,4.0,The Horseman's Ale,5.2,33061,Pumpkin Ale,None,The Defiant Brewing Company Pumpkin Ale Pumpki...
1586612,14359,The Defiant Brewing Company,1160023044,4.0,4.5,4.5,dherling,4.5,4.5,The Horseman's Ale,5.2,33061,Pumpkin Ale,None,The Defiant Brewing Company Pumpkin Ale Pumpki...


Birazdan bira tavsiye sistemimizi kurduktan sonra, aynı özellikte olan **biraları birden fazla defa tavsiye etmesini istemeyiz**. Bu nedenle, `attributes` sütunundaki tekrar eden değerlerden kurtulalım.

In [14]:
df_new.drop_duplicates(subset=['attributes'], inplace=True)
df_new.drop_duplicates(subset=['beer_name'], inplace=True)
df_new.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 36201 entries, 0 to 1586595
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   brewery_id          36201 non-null  int64  
 1   brewery_name        36200 non-null  object 
 2   review_time         36201 non-null  int64  
 3   review_overall      36201 non-null  float64
 4   review_aroma        36201 non-null  float64
 5   review_appearance   36201 non-null  float64
 6   review_profilename  36191 non-null  object 
 7   review_palate       36201 non-null  float64
 8   review_taste        36201 non-null  float64
 9   beer_name           36201 non-null  object 
 10  beer_abv            27361 non-null  float64
 11  beer_beerid         36201 non-null  int64  
 12  1st_beer_style      36201 non-null  object 
 13  2nd_beer_style      7268 non-null   object 
 14  attributes          36200 non-null  object 
dtypes: float64(6), int64(3), object(6)
memory usage: 4.

In [15]:
df_new.dropna(subset='brewery_name', inplace=True)
df_new.reset_index(drop=True, inplace=True)
df_new

,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,review_palate,review_taste,beer_name,beer_abv,beer_beerid,1st_beer_style,2nd_beer_style,attributes
0,10325,Vecchio Birraio,1234817823,1.5,2.0,2.5,stcules,1.5,1.5,Sausa Weizen,5.0,47986,Hefeweizen,None,Vecchio Birraio Hefeweizen Hefeweizen
1,10325,Vecchio Birraio,1235915097,3.0,2.5,3.0,stcules,3.0,3.0,Red Moon,6.2,48213,English Strong Ale,None,Vecchio Birraio English Strong Ale English Str...
2,10325,Vecchio Birraio,1235916604,3.0,2.5,3.0,stcules,3.0,3.0,Black Horse Black Beer,6.5,48215,Foreign,Export Stout,Vecchio Birraio Foreign Foreign
3,10325,Vecchio Birraio,1234725145,3.0,3.0,3.5,stcules,2.5,3.0,Sausa Pils,5.0,47969,German Pilsener,None,Vecchio Birraio German Pilsener German Pilsener
4,1075,Caldera Brewing Company,1293735206,4.0,4.5,4.0,johnmichaelsen,4.0,4.5,Cauldron DIPA,7.7,64883,American Double,Imperial IPA,Caldera Brewing Company American Double Ameri...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36195,14359,The Defiant Brewing Company,1268076132,4.0,4.0,4.0,plaid75,4.0,4.0,Prohibition Lager,NaN,49776,American Pale Lager,None,The Defiant Brewing Company American Pale Lage...
36196,14359,The Defiant Brewing Company,1263965669,4.0,4.0,4.0,ClockworkOrange,4.0,4.0,2007 Resolution #1,7.4,34405,Belgian Strong Pale Ale,None,The Defiant Brewing Company Belgian Strong Pal...
36197,14359,The Defiant Brewing Company,1305153333,4.0,3.5,4.5,plaid75,4.5,4.5,O'Defiant Stout,5.5,36388,Irish Dry Stout,None,The Defiant Brewing Company Irish Dry Stout Ir...
36198,14359,The Defiant Brewing Company,1295399777,2.0,2.5,3.0,Buddha22,2.0,2.5,Bear Mountain Ale,8.0,62147,Belgian IPA,None,The Defiant Brewing Company Belgian IPA Belgia...


Benzer özellikteki biraları bulmak için `cosine_similarity()` yönteminden yararlanacağız. Son hazırlıklarımızı da tamamlayalım...

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english') # TF-IDF yöntemini kullanalım
df_new['attributes'] = df_new['attributes'].fillna('')

tfidf_matrix = tfidf.fit_transform(df_new['attributes'])
tfidf_matrix.shape

(36200, 6491)

In [17]:
from sklearn.metrics.pairwise import cosine_similarity # Hesaplaması biraz zaman alabilir
cosine_sim = cosine_similarity(tfidf_matrix)
cosine_sim

array([[1.        , 0.63930746, 0.60578263, ..., 0.        , 0.        ,
        0.3902061 ],
       [0.63930746, 1.        , 0.55629998, ..., 0.        , 0.        ,
        0.        ],
       [0.60578263, 0.55629998, 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.29740721,
        0.33352635],
       [0.        , 0.        , 0.        , ..., 0.29740721, 1.        ,
        0.44483249],
       [0.3902061 , 0.        , 0.        , ..., 0.33352635, 0.44483249,
        1.        ]])

Biralar arasındaki benzerlikleri hesapladık. Ancak bunu daha düzgün çıktıda sunmamız lazım, bize direkt **en benzer biraların isimleri** lazım.

In [18]:
indices = pd.Series(df_new.index, index=df_new['beer_name'])
indices

beer_name
Sausa Weizen                  0
Red Moon                      1
Black Horse Black Beer        2
Sausa Pils                    3
Cauldron DIPA                 4
                          ...  
Prohibition Lager         36195
2007 Resolution #1        36196
O'Defiant Stout           36197
Bear Mountain Ale         36198
Baron Von Weizen          36199
Length: 36200, dtype: int64

Son olarak da kullancılara tavsiyede bulunmak için kullanacağımız fonksiyonumuzu hazırlayalım.

In [19]:
def get_recommendations(beer_name, cosine_sim=cosine_sim):
    
    idx = indices[beer_name] # Her bir biraya karşılık gelen index değerleri

    sim_scores = list(enumerate(cosine_sim[idx])) # Biralar arasındaki ikili benzerlik puanları

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True) # Benzerlik oranlarına göre sıralama

    sim_scores = sim_scores[1:11] # En benzer 10 bira

    beer_indices = [i[0] for i in sim_scores] # Bu biraların index değerleri

    return df_new['beer_name'].iloc[beer_indices] # En benzer 10 birayı göster

In [20]:
get_recommendations('Red Moon')

0                                      Sausa Weizen
20734                    Dunder & Blixem Strong Ale
3                                        Sausa Pils
2                            Black Horse Black Beer
8500                         Albert Anderson (A.A.)
33504                                    Otter Head
34894                          Brooklyn Backbreaker
23067                                     Old Cedar
33156                            The Shropshire Way
16339    House Ales St. Andre Wozniak Part-Gyle 2.0
Name: beer_name, dtype: object

In [21]:
get_recommendations('Irish Amber')

20721      Brown's Irish Amber Ale
14503           Red Rock Irish Ale
10068         Barley's Irish Rogue
1065        Wolf's Irish Pub Draft
20392                  Tiger Shark
35922             King's Irish Red
4940                        Redrum
12392         People's 9 Irish Red
18530        Saranac Irish Red Ale
16700    Celtic Knot Irish Red Ale
Name: beer_name, dtype: object

In [22]:
get_recommendations('Black Horse Black Beer')

30379                            Tinker Stout
0                                Sausa Weizen
8478                  Bootblack's Extra Stout
25393                              Naomi Dyal
20741        Phil N. Topemov's Imperial Stout
3639                            Jamaica Stout
3921     Arbor Brewing St. Pat's Strong Stout
18932           Kuhnhenn Foreign Export Stout
23290                 Pale Horse Export Stout
17238                Mr. Miagi's Wasabi Stout
Name: beer_name, dtype: object

## 2. Collaborative Filtering Tavsiye Sistemleri: Biralar

Benzer ürünleri bularak onlar üzerinden tavsiyelerimizi sunmak oldukça mantıklı bir yaklaşım tarzı. Ancak burada şöyle bir sorun var.

> Gerçekten **sadece benzer ürünleri ve içerikleri** mi bulmak isteriz?

Cevap tabiki **HAYIR!** İnsanlar birbirleriyle doğrudan bir ilgisi olmasa bile **farklı tarzda ürünlere** de oldukça ilgi gösterebiliyorlar. Peki bu nasıl bulabiliriz?

> Evet, **müşteri davranışlarını takip ederek!**

**Hikaye Linki:** https://canworksmart.com/diapers-beer-retail-predictive-analytics/

![diapers_beer_correlation](https://i0.wp.com/canworksmart.com/wp-content/uploads/2012/07/DIAPERS-VS-BEER.jpg?fit=1024%2C684&ssl=1)

Oldukça büyük bir veri setimiz olduğu için şimdilik işlem hacmini biraz daha az tutmak için, bu çalışmada **en çok değerlendirilen 250 bira adını** kullanalım.

In [23]:
df.beer_name.value_counts()

90 Minute IPA                          3290
India Pale Ale                         3130
Old Rasputin Russian Imperial Stout    3111
Sierra Nevada Celebration Ale          3000
Two Hearted Ale                        2728
                                       ... 
Titanbraü Pils                            1
Titanbraü Weizen                          1
Titanbräu Ale                             1
Texas Brunette                            1
Sausa Weizen                              1
Name: beer_name, Length: 56857, dtype: int64

In [24]:
n = 250
top_n = df.beer_name.value_counts().index[:n] # En çok değerlendirilen 250 bira

df = df[df['beer_name'].isin(top_n)]
df.head()

,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,review_palate,review_taste,beer_name,beer_abv,beer_beerid,1st_beer_style,2nd_beer_style
798,1075,Caldera Brewing Company,1212201268,4.5,4.5,4.0,grumpy,4.0,4.5,Imperial Stout,NaN,42964,American Double,Imperial Stout
1559,11715,Destiny Brewing Company,1137124057,4.0,3.5,4.0,blitheringidiot,3.5,3.5,Pale Ale,4.5,26420,American Pale Ale (APA),None
1560,11715,Destiny Brewing Company,1129504403,4.0,2.5,4.0,NeroFiddled,4.0,3.5,Pale Ale,4.5,26420,American Pale Ale (APA),None
1563,11715,Destiny Brewing Company,1137125989,3.5,3.0,4.0,blitheringidiot,4.0,4.0,IPA,NaN,26132,American IPA,None
1564,11715,Destiny Brewing Company,1130936611,3.0,3.0,3.0,Gavage,4.0,3.5,IPA,NaN,26132,American IPA,None


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 355275 entries, 798 to 1586564
Data columns (total 14 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   brewery_id          355275 non-null  int64  
 1   brewery_name        355275 non-null  object 
 2   review_time         355275 non-null  int64  
 3   review_overall      355275 non-null  float64
 4   review_aroma        355275 non-null  float64
 5   review_appearance   355275 non-null  float64
 6   review_profilename  355175 non-null  object 
 7   review_palate       355275 non-null  float64
 8   review_taste        355275 non-null  float64
 9   beer_name           355275 non-null  object 
 10  beer_abv            353477 non-null  float64
 11  beer_beerid         355275 non-null  int64  
 12  1st_beer_style      355275 non-null  object 
 13  2nd_beer_style      90617 non-null   object 
dtypes: float64(6), int64(3), object(5)
memory usage: 40.7+ MB


> **SORU:** Sizce benzer müşteri davranışlarını bulmak için hangi sütunlarla çalışabiliriz?

Buradaki ana amacımız bu sefer kullanıcı davranışları arasındaki benzerlikleri bulmak. Yani bu amaç için de `cosine_similarity()` yönteminden yararlanabiliriz. Ama tabiki öncelikle kullanacağımız veriyi hazırlamamız gerekiyor.

In [26]:
df.describe()

,brewery_id,review_time,review_overall,review_aroma,review_appearance,review_palate,review_taste,beer_abv,beer_beerid
count,355275.000000,3.552750e+05,355275.000000,355275.00000,355275.000000,355275.000000,355275.000000,353477.000000,355275.000000
mean,1438.567166,1.215667e+09,3.997600,3.92471,4.012851,3.935513,4.019834,7.415435,8004.432068
std,3441.850525,7.580421e+07,0.678365,0.67695,0.595382,0.652733,0.696001,2.339414,12726.636998
min,1.000000,8.867232e+08,1.000000,1.00000,1.000000,1.000000,1.000000,2.500000,30.000000
25%,105.000000,1.164174e+09,3.500000,3.50000,4.000000,3.500000,3.500000,5.600000,572.000000
50%,192.000000,1.229493e+09,4.000000,4.00000,4.000000,4.000000,4.000000,7.000000,1481.000000
75%,664.000000,1.278640e+09,4.500000,4.50000,4.500000,4.500000,4.500000,9.000000,9478.000000
max,27980.000000,1.326274e+09,5.000000,5.00000,5.000000,5.000000,5.000000,18.000000,77247.000000


In [27]:
df_wide = pd.pivot_table(df, values=['review_overall'],
        index=['beer_name', 'review_profilename'],
        aggfunc=np.mean).unstack() # Tablonun görünümünü düzenlemek için küçük bir düzeltme
df_wide.shape

(250, 22140)

Bakalım nasıl bir DataFrame oluşturmuşuz?

In [28]:
df_wide

review_overall                               \
review_profilename                   0110x011 02maxima 03SVTCobra 05Harley   
beer_name                                                                    
#9                                        NaN      NaN        NaN      NaN   
120 Minute IPA                            NaN      NaN        NaN      4.0   
1554 Enlightened Black Ale                NaN      NaN        NaN      NaN   
60 Minute IPA                             NaN      NaN        NaN      NaN   
90 Minute IPA                             5.0      NaN        NaN      4.0   
...                                       ...      ...        ...      ...   
World Wide Stout                          NaN      NaN        NaN      4.0   
Yeti Imperial Stout                       NaN      NaN        NaN      NaN   
Young's Double Chocolate Stout            NaN      NaN        NaN      NaN   
Yuengling Traditional Lager               NaN      NaN        NaN      NaN   
Éphémère (Apple)                          NaN      NaN        NaN      NaN   

                                                                             \
review_profilename             0Naught0 0beerguy0 0runkp0s 0tt0 1000Bottles   
beer_name                                                                     
#9                                  NaN       NaN      NaN  NaN         NaN   
120 Minute IPA                      NaN       NaN      NaN  1.5         NaN   
1554 Enlightened Black Ale          NaN       NaN      NaN  NaN         NaN   
60 Minute IPA                       NaN       NaN      NaN  NaN         NaN   
90 Minute IPA                       NaN       NaN      NaN  NaN         NaN   
...                                 ...       ...      ...  ...         ...   
World Wide Stout                    NaN       NaN      NaN  NaN         NaN   
Yeti Imperial Stout                 NaN       NaN      NaN  NaN         NaN   
Young's Double Chocolate Stout      NaN       NaN      NaN  NaN         NaN   
Yuengling Traditional Lager         NaN       NaN      NaN  NaN         NaN   
Éphémère (Apple)                    NaN       NaN      NaN  NaN         NaN   

                                          ...                             \
review_profilename             1001111.0  ... zuker zulufactor zumicroom   
beer_name                                 ...                              
#9                                   NaN  ...   NaN        NaN       NaN   
120 Minute IPA                       NaN  ...   NaN        NaN       NaN   
1554 Enlightened Black Ale           NaN  ...   NaN        NaN       NaN   
60 Minute IPA                        NaN  ...   NaN        NaN       NaN   
90 Minute IPA                        NaN  ...   NaN        NaN       NaN   
...                                  ...  ...   ...        ...       ...   
World Wide Stout                     NaN  ...   NaN        NaN       NaN   
Yeti Imperial Stout                  NaN  ...   NaN        NaN       NaN   
Young's Double Chocolate Stout       NaN  ...   NaN        NaN       NaN   
Yuengling Traditional Lager          NaN  ...   NaN        NaN       NaN   
Éphémère (Apple)                     NaN  ...   NaN        NaN       NaN   

                                                                             \
review_profilename             zwalk8 zwoehr zymrgy zymurgy4all zymurgywhiz   
beer_name                                                                     
#9                                NaN    NaN    NaN         NaN         NaN   
120 Minute IPA                    NaN    NaN    NaN         NaN         NaN   
1554 Enlightened Black Ale        NaN    NaN    NaN         NaN         NaN   
60 Minute IPA                     NaN    NaN    NaN         NaN         NaN   
90 Minute IPA                     NaN    NaN    NaN         NaN         NaN   
...                               ...    ...    ...         ...         ...   
World Wide Stout                  NaN    NaN    3.0    

> **SORU:** Oluşturduğumuz DataFrame'de bir sürü **NaN** değer var. Bunlar nereden gelmiş olabilir?

Kullanıcıların davranış benzerliğine bakmak istediğimizde **NaN verilerle çalışamayız**. Dolayısıyla, etkisini en az düzeyde tutmak için DataFrame'imizdeki **tüm NaN olan değerleri 0'a** atayalım.

In [29]:
df_wide = df_wide.fillna(0)
df_wide

review_overall                               \
review_profilename                   0110x011 02maxima 03SVTCobra 05Harley   
beer_name                                                                    
#9                                        0.0      0.0        0.0      0.0   
120 Minute IPA                            0.0      0.0        0.0      4.0   
1554 Enlightened Black Ale                0.0      0.0        0.0      0.0   
60 Minute IPA                             0.0      0.0        0.0      0.0   
90 Minute IPA                             5.0      0.0        0.0      4.0   
...                                       ...      ...        ...      ...   
World Wide Stout                          0.0      0.0        0.0      4.0   
Yeti Imperial Stout                       0.0      0.0        0.0      0.0   
Young's Double Chocolate Stout            0.0      0.0        0.0      0.0   
Yuengling Traditional Lager               0.0      0.0        0.0      0.0   
Éphémère (Apple)                          0.0      0.0        0.0      0.0   

                                                                             \
review_profilename             0Naught0 0beerguy0 0runkp0s 0tt0 1000Bottles   
beer_name                                                                     
#9                                  0.0       0.0      0.0  0.0         0.0   
120 Minute IPA                      0.0       0.0      0.0  1.5         0.0   
1554 Enlightened Black Ale          0.0       0.0      0.0  0.0         0.0   
60 Minute IPA                       0.0       0.0      0.0  0.0         0.0   
90 Minute IPA                       0.0       0.0      0.0  0.0         0.0   
...                                 ...       ...      ...  ...         ...   
World Wide Stout                    0.0       0.0      0.0  0.0         0.0   
Yeti Imperial Stout                 0.0       0.0      0.0  0.0         0.0   
Young's Double Chocolate Stout      0.0       0.0      0.0  0.0         0.0   
Yuengling Traditional Lager         0.0       0.0      0.0  0.0         0.0   
Éphémère (Apple)                    0.0       0.0      0.0  0.0         0.0   

                                          ...                             \
review_profilename             1001111.0  ... zuker zulufactor zumicroom   
beer_name                                 ...                              
#9                                   0.0  ...   0.0        0.0       0.0   
120 Minute IPA                       0.0  ...   0.0        0.0       0.0   
1554 Enlightened Black Ale           0.0  ...   0.0        0.0       0.0   
60 Minute IPA                        0.0  ...   0.0        0.0       0.0   
90 Minute IPA                        0.0  ...   0.0        0.0       0.0   
...                                  ...  ...   ...        ...       ...   
World Wide Stout                     0.0  ...   0.0        0.0       0.0   
Yeti Imperial Stout                  0.0  ...   0.0        0.0       0.0   
Young's Double Chocolate Stout       0.0  ...   0.0        0.0       0.0   
Yuengling Traditional Lager          0.0  ...   0.0        0.0       0.0   
Éphémère (Apple)                     0.0  ...   0.0        0.0       0.0   

                                                                             \
review_profilename             zwalk8 zwoehr zymrgy zymurgy4all zymurgywhiz   
beer_name                                                                     
#9                                0.0    0.0    0.0         0.0         0.0   
120 Minute IPA                    0.0    0.0    0.0         0.0         0.0   
1554 Enlightened Black Ale        0.0    0.0    0.0         0.0         0.0   
60 Minute IPA                     0.0    0.0    0.0         0.0         0.0   
90 Minute IPA                     0.0    0.0    0.0         0.0         0.0   
...                               ...    ...    ...         ...         ...   
World Wide Stout                  0.0    0.0    3.0    

Artık kullanıcı davranışları arasındaki benzerliğe bakabiliriz.

>**SORU:** 2 ürün veya davranışın birbiriyle olan benzerliklerini sizce nasıl tespit ediyoruz?

Bu yöntemlerin birçok çeşidi var. Biz şu ana kadar pek mantığını bilmesek de **cosine similarity** kullandık :) Bakalım başka neler varmış?

- **Cosine Similarity** (En Çok Tercih Edilen)
- **Euclidean Distance**
- **Manhattan Distance**

>**HATIRLATMA:** cos(0°) = 1, cos(90°) = 0

![distance_metrics](https://dh2016.adho.org/abstracts/static/data/290/10000201000007AF000007CFCCC81279FE2EA7FD.png)

In [30]:
cosine_sim2 = cosine_similarity(df_wide)
cosine_sim2

array([[1.        , 0.27540494, 0.27410345, ..., 0.32928048, 0.34805798,
        0.31249922],
       [0.27540494, 1.        , 0.25151873, ..., 0.2854835 , 0.23301356,
        0.2802485 ],
       [0.27410345, 0.25151873, 1.        , ..., 0.31629515, 0.22521858,
        0.2737628 ],
       ...,
       [0.32928048, 0.2854835 , 0.31629515, ..., 1.        , 0.28025764,
        0.34504013],
       [0.34805798, 0.23301356, 0.22521858, ..., 0.28025764, 1.        ,
        0.25526913],
       [0.31249922, 0.2802485 , 0.2737628 , ..., 0.34504013, 0.25526913,
        1.        ]])

In [31]:
cosine_sim2_df = pd.DataFrame(cosine_sim2, index=df_wide.index, columns=df_wide.index)
cosine_sim2_df.head()

beer_name,#9,120 Minute IPA,1554 Enlightened Black Ale,60 Minute IPA,90 Minute IPA,Aecht Schlenkerla Rauchbier Märzen,AleSmith IPA,AleSmith Speedway Stout,Allagash White,Alpha King Pale Ale,...,Vanilla Porter,Weihenstephaner Hefeweissbier,Weihenstephaner Korbinian,Westmalle Trappist Dubbel,Westmalle Trappist Tripel,World Wide Stout,Yeti Imperial Stout,Young's Double Chocolate Stout,Yuengling Traditional Lager,Éphémère (Apple)
beer_name,,,,,,,,,,,,,,,,,,,,,
#9,1.000000,0.275405,0.274103,0.388364,0.365175,0.253841,0.228479,0.227612,0.340681,0.293315,...,0.266570,0.312395,0.276463,0.233554,0.276763,0.286534,0.299032,0.329280,0.348058,0.312499
120 Minute IPA,0.275405,1.000000,0.251519,0.378258,0.410366,0.262425,0.315971,0.337541,0.282273,0.336796,...,0.201428,0.312193,0.282320,0.270800,0.301144,0.418214,0.337978,0.285483,0.233014,0.280248
1554 Enlightened Black Ale,0.274103,0.251519,1.000000,0.319887,0.314028,0.252486,0.266866,0.261761,0.260275,0.307296,...,0.285846,0.300474,0.292369,0.265445,0.271656,0.262771,0.295029,0.316295,0.225219,0.273763
60 Minute IPA,0.388364,0.378258,0.319887,1.000000,0.533042,0.316928,0.312343,0.307627,0.360975,0.385249,...,0.285143,0.413405,0.329941,0.308774,0.355926,0.358224,0.391041,0.399840,0.326916,0.339324
90 Minute IPA,0.365175,0.410366,0.314028,0.533042,1.000000,0.312861,0.344218,0.358754,0.356804,0.418582,...,0.262775,0.436398,0.343738,0.333099,0.387312,0.405116,0.414385,0.395031,0.301877,0.332292


In [32]:
from sklearn.metrics.pairwise import euclidean_distances

dist_euc = euclidean_distances(df_wide)
dist_euc

array([[  0.        , 174.22993269, 163.58021308, ..., 194.84924728,
        161.74899107, 147.03771719],
       [174.22993269,   0.        , 176.40826228, ..., 207.65394223,
        185.16205365, 161.91799622],
       [163.58021308, 176.40826228,   0.        , ..., 196.12897896,
        175.58968111, 150.19716209],
       ...,
       [194.84924728, 207.65394223, 196.12897896, ...,   0.        ,
        205.71759567, 185.28360394],
       [161.74899107, 185.16205365, 175.58968111, ..., 205.71759567,
          0.        , 160.24813658],
       [147.03771719, 161.91799622, 150.19716209, ..., 185.28360394,
        160.24813658,   0.        ]])

In [33]:
dist_euc_df = pd.DataFrame(dist_euc, index=df_wide.index, columns=df_wide.index)
dist_euc_df.head()

beer_name,#9,120 Minute IPA,1554 Enlightened Black Ale,60 Minute IPA,90 Minute IPA,Aecht Schlenkerla Rauchbier Märzen,AleSmith IPA,AleSmith Speedway Stout,Allagash White,Alpha King Pale Ale,...,Vanilla Porter,Weihenstephaner Hefeweissbier,Weihenstephaner Korbinian,Westmalle Trappist Dubbel,Westmalle Trappist Tripel,World Wide Stout,Yeti Imperial Stout,Young's Double Chocolate Stout,Yuengling Traditional Lager,Éphémère (Apple)
beer_name,,,,,,,,,,,,,,,,,,,,,
#9,0.000000,174.229933,163.580213,198.889172,226.158606,154.688688,167.027568,174.317897,160.150862,182.979113,...,147.634684,204.219568,159.651269,166.965524,179.609434,168.944168,173.414384,194.849247,161.748991,147.037717
120 Minute IPA,174.229933,0.000000,176.408262,205.519464,223.010510,165.243948,167.371136,170.681335,176.769447,185.167028,...,166.088748,210.549727,169.514011,173.160691,185.098454,161.099154,177.082431,207.653942,185.162054,161.917996
1554 Enlightened Black Ale,163.580213,176.408262,0.000000,207.994591,233.043049,154.017247,162.097236,169.752618,168.938783,180.671880,...,144.922297,205.378462,157.160746,162.734830,179.630698,171.059383,173.295809,196.128979,175.589681,150.197162
60 Minute IPA,198.889172,205.519464,207.994591,0.000000,216.172240,202.668294,208.132603,212.805744,204.677338,210.682114,...,203.470650,220.821902,204.694895,208.868679,212.163704,206.353610,204.649243,218.587114,211.036595,199.301123
90 Minute IPA,226.158606,223.010510,233.043049,216.172240,0.000000,228.835640,228.252038,228.923871,228.818979,226.068608,...,232.353165,234.272544,227.533377,229.965894,229.054306,222.394638,223.130287,238.551823,238.051742,225.997788


In [34]:
from sklearn.metrics.pairwise import manhattan_distances

dist_mht = manhattan_distances(df_wide)
dist_mht

array([[    0.        ,  8302.33333333,  7114.83333333, ...,
         9759.08333333,  7028.83333333,  5978.58333333],
       [ 8302.33333333,     0.        ,  8189.        , ...,
        10996.25      ,  9040.        ,  7125.75      ],
       [ 7114.83333333,  8189.        ,     0.        , ...,
         9499.58333333,  7864.33333333,  5896.25      ],
       ...,
       [ 9759.08333333, 10996.25      ,  9499.58333333, ...,
            0.        , 10558.58333333,  8652.83333333],
       [ 7028.83333333,  9040.        ,  7864.33333333, ...,
        10558.58333333,     0.        ,  6758.25      ],
       [ 5978.58333333,  7125.75      ,  5896.25      , ...,
         8652.83333333,  6758.25      ,     0.        ]])

In [35]:
dist_mht_df = pd.DataFrame(dist_mht, index=df_wide.index, columns=df_wide.index)
dist_mht_df.head()

beer_name,#9,120 Minute IPA,1554 Enlightened Black Ale,60 Minute IPA,90 Minute IPA,Aecht Schlenkerla Rauchbier Märzen,AleSmith IPA,AleSmith Speedway Stout,Allagash White,Alpha King Pale Ale,...,Vanilla Porter,Weihenstephaner Hefeweissbier,Weihenstephaner Korbinian,Westmalle Trappist Dubbel,Westmalle Trappist Tripel,World Wide Stout,Yeti Imperial Stout,Young's Double Chocolate Stout,Yuengling Traditional Lager,Éphémère (Apple)
beer_name,,,,,,,,,,,,,,,,,,,,,
#9,0.000000,8302.333333,7114.833333,10054.666667,12817.083333,6444.083333,7177.666667,7749.916667,6697.083333,8453.041667,...,6061.000000,10070.916667,6682.166667,7241.833333,8237.583333,7623.083333,7826.083333,9759.083333,7028.833333,5978.583333
120 Minute IPA,8302.333333,0.000000,8189.000000,10785.000000,12569.250000,7283.750000,7265.500000,7525.750000,8036.750000,8719.375000,...,7504.666667,10776.750000,7492.500000,7798.500000,8752.750000,7025.250000,8195.250000,10996.250000,9040.000000,7125.750000
1554 Enlightened Black Ale,7114.833333,8189.000000,0.000000,10521.500000,13170.250000,6038.250000,6425.500000,7015.250000,6972.250000,7853.375000,...,5555.333333,9793.250000,6124.000000,6536.500000,7858.750000,7487.250000,7475.250000,9499.583333,7864.333333,5896.250000
60 Minute IPA,10054.666667,10785.000000,10521.500000,0.000000,11577.250000,10006.250000,10338.500000,10797.583333,10064.250000,10633.125000,...,10223.500000,11525.250000,10083.000000,10488.500000,10814.250000,10565.750000,10208.250000,11683.083333,10989.166667,9802.250000
90 Minute IPA,12817.083333,12569.250000,13170.250000,11577.250000,0.000000,12702.000000,12498.750000,12597.333333,12608.000000,12363.125000,...,13239.083333,13131.000000,12467.750000,12728.750000,12678.500000,12249.500000,12121.500000,13907.833333,13874.083333,12510.000000


Bu örnekte işleri biraz daha zorlaştıralım.

İlk çalışmamızda önerilerimizi yaparken kullanıcıdan bize **sevdiği birayı söylemesini** ve bizim elimizdeki bilgileri kullanarak ona **en benzer hangi birayı** tavsiye edebileceğimiz örneğini işlemiştik. 


Şimdi ise kullanıcının en sevdiği 3 birayı öğrenip, **bu 3 birayı da göz önünde bulundurarak** çeşitli tavsiyelerde bulunalım.

>**HATIRLATMA:** Bu çalışmamızda kullanıcı davranışlarına göre tavsiyede bulunacağımızı unutmayın.

In [36]:
beers_i_like = ['Sierra Nevada Pale Ale', '120 Minute IPA', 'Allagash White'] # Sizi seçtik pikachular :)
cosine_sim2_df[beers_i_like].head() # İlk olarak cosine-similarity

beer_name,Sierra Nevada Pale Ale,120 Minute IPA,Allagash White
beer_name,,,
#9,0.373968,0.275405,0.340681
120 Minute IPA,0.301693,1.000000,0.282273
1554 Enlightened Black Ale,0.330033,0.251519,0.260275
60 Minute IPA,0.459641,0.378258,0.360975
90 Minute IPA,0.441189,0.410366,0.356804


Bu çalışmada 3 tane birayı göz önünde bulundurarak bir tavsiyede bulunacağımızı konuşmuştuk. Bunu yapmak için, basit bir mantıkla **her bir biradan elde ettiğimiz benzerlik oranlarını kendi içlerinde toplayıp**, en yüksek benzerlik oranına ulaştığımız birayı şampiyon olarak seçebiliriz.

Bunu yapmanın 2 yolu var:

1. Oluşturduğumuz DataFrame'de `apply()` fonksiyonunu kullanarak toplamı hesaplamak
2. Ya da `np.sum()` fonksiyonunu kullanarak seçtiğimiz biralardaki benzerlik oranlarını toplamak

>**SORU** Sizce hangisi yöntem işimizi daha hızlı yapmamızı sağlar?

In [37]:
%timeit cosine_sim2_df[beers_i_like].apply(lambda row: np.sum(row), axis=1)

8.73 ms ± 139 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [38]:
%timeit np.sum(cosine_sim2_df[beers_i_like], axis=1)

303 µs ± 2.93 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [39]:
beers_summed = np.sum(cosine_sim2_df[beers_i_like], axis=1)

En benzerlerini ilk satırlarda görmek için sıralamayı değiştirelim.

In [40]:
beers_summed = beers_summed.sort_values(ascending=False)
beers_summed

beer_name
Sierra Nevada Pale Ale           1.654205
Allagash White                   1.634784
120 Minute IPA                   1.583966
HopDevil Ale                     1.224217
Sierra Nevada Celebration Ale    1.215156
                                   ...   
Bud Light                        0.710377
Corona Extra                     0.682882
Miller High Life                 0.682360
Coors Light                      0.670167
Vanilla Porter                   0.664363
Length: 250, dtype: float64

Sonuçları daha güzel bir şekilde görmek için buraya kadar konuştuklarımızdan bir fonksiyon oluşturalım.

In [41]:
def find_similar_beers(beers, count=1):
    """
    Parameters
    ----------
    beers: list
        some beer names!
        
    count: int (default=1)
        count of similar beer!
    
    Returns
    -------
    ranked_beers: list
        rank ordered beers
    """
    beers = [beer for beer in beers if beer in cosine_sim2_df.columns]
    beers_summed = cosine_sim2_df[beers].apply(lambda row: np.sum(row), axis=1)
    beers_summed = beers_summed.sort_values(ascending=False) # Yüksek skorlar daha iyi
    ranked_beers = beers_summed.index[beers_summed.index.isin(beers)==False]
    ranked_beers = ranked_beers.tolist()

    if count is None:
        return ranked_beers
    else:
        return ranked_beers[:count]

Şimdi de fonksiyonumuzu test edelim.

In [42]:
for beer in find_similar_beers(["120 Minute IPA"], 10):
    print(beer)

World Wide Stout
90 Minute IPA
Double Bastard Ale
Stone Ruination IPA
Stone Imperial Russian Stout
Storm King Stout
60 Minute IPA
Oaked Arrogant Bastard Ale
Sierra Nevada Bigfoot Barleywine Style Ale
Brooklyn Black Chocolate Stout


In [43]:
def find_similar_beers2(beers, dist_metric, count=1):
    """
    Parameters
    ----------
    beers: list
        some beer names!
        
    dist_metric: variable
        distance metric u prefer to use (must be: euclidean or manhattan distance)
        
    count: int (default=1)
        count of similar beer!
    
    Returns
    -------
    ranked_beers: list
        rank ordered beers
    """
    beers = [beer for beer in beers if beer in cosine_sim2_df.columns]
    beers_summed = cosine_sim2_df[beers].apply(lambda row: np.sum(row), axis=1)
    beers_summed = beers_summed.sort_values() # Düşük skorlar daha iyi
    ranked_beers = beers_summed.index[beers_summed.index.isin(beers)==False]
    ranked_beers = ranked_beers.tolist()

    if count is None:
        return ranked_beers
    else:
        return ranked_beers[:count]

In [44]:
for beer in find_similar_beers2(['120 Minute IPA'], dist_euc_df, 10):
    print(beer)

Miller High Life
Corona Extra
Coors Light
Miller Lite
Bud Light
Murphy's Irish Stout
Budweiser
Vanilla Porter
Hobgoblin
Shiner Bock


In [45]:
for beer in find_similar_beers2(['120 Minute IPA'], dist_mht_df, 10):
    print(beer)

Miller High Life
Corona Extra
Coors Light
Miller Lite
Bud Light
Murphy's Irish Stout
Budweiser
Vanilla Porter
Hobgoblin
Shiner Bock


Peki **3 farklı birayı aynı anda** girdi olarak verirsek?

In [46]:
for number, beer in enumerate(find_similar_beers(['Coors Light', 'Bud Light', 'Amstel Light'], 5)):
    print('%d) %s' % (number+1, beer))

1) Miller Lite
2) Budweiser
3) Corona Extra
4) Samuel Adams Boston Lager
5) Heineken Lager Beer


In [47]:
for number, beer in enumerate(find_similar_beers2(['Coors Light', 'Bud Light', 'Amstel Light'], dist_euc_df, 5)):
    print('%d) %s' % (number+1, beer))

1) Supplication
2) The Abyss
3) Founders Backwoods Bastard
4) New Holland Dragon's Milk Oak Barrel Ale
5) Terrapin Coffee Oatmeal Imperial Stout


In [48]:
for number, beer in enumerate(find_similar_beers2(['Coors Light', 'Bud Light', 'Amstel Light'], dist_mht_df, 5)):
    print('%d) %s' % (number+1, beer))

1) Supplication
2) The Abyss
3) Founders Backwoods Bastard
4) New Holland Dragon's Milk Oak Barrel Ale
5) Terrapin Coffee Oatmeal Imperial Stout
